[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_4/L1.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 4 | BÀI 1


---

# **HỌC TẬP KẾT HỢP (ENSEMBLE LEARNING)**

|  |  |
|:---|:---|
|**Thời gian đọc** | 18 phút |
|**Kiến thức nền** | Naive Bayes, máy vector hỗ trợ, cây quyết định, tập huấn luyện và tập kiểm tra.  |
|**Từ khóa** |Blending, Bagging, Stacking, kiểm định.  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_4/"
FILES = ['meta_dat2train.csv']
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
        print("Đã tải:", f)

*Đến giờ, ta đã xem nhiều kỹ thuật học máy, cụ thể là các bộ phân loại. Ta thường tìm nhiều ý kiến rồi lấy đa số để định hướng quyết định cuối cùng. Hãy nghĩ đến các đánh giá/phản hồi về một sản phẩm quảng cáo trực tuyến. Học tập kết hợp hoạt động theo cách tương tự. Quyết định từ nhiều bộ phân loại được kết hợp, và kết quả cuối cùng là bỏ phiếu đa số hoặc trung bình có trọng số. Cách này có một số lợi ích như cải thiện độ chính xác, giảm sai số khái quát hóa của từng bộ phân loại (Opitz và Maclin 169) và giảm sai số phương sai (Ali và Pazzani 173) mà không làm hại độ chệch. Ta đã nhắc đến học tập kết hợp ở Module 3 khi nói về boosting, ý tưởng kết hợp các bộ học yếu thành một bộ học mạnh. Ở phần tiếp theo, ta xem một phương pháp kết hợp, cụ thể là **Bagging**.*

In [1]:
import pandas as pd

## **1. Bagging**

Bagging hay bootstrap aggregation (tập hợp mẫu có hoàn lại) là cách giảm phương sai trong một tập dữ liệu nhiễu. Trong quy trình này, các bộ phân loại yếu được huấn luyện độc lập trên một mẫu thay vì toàn bộ tập huấn luyện. Các mẫu này không nhất thiết nhỏ hơn tập huấn luyện; tuy nhiên kích thước mẫu thường bằng tập huấn luyện để mỗi bộ phân loại có đủ dữ liệu để học. Mẫu được lấy có hoàn lại; do đó một quan sát có thể đến với nhiều bộ phân loại và một mẫu có thể chứa một quan sát nhiều lần. Các bộ phân loại yếu không nhất thiết có cùng biến dự báo hay đặc trưng. Mỗi bộ phân loại yếu nhiều khả năng sẽ có cách dự báo nhóm mục tiêu khác nhau, không chỉ trên các đặc trưng khác nhau mà còn ở những điểm chia khác nhau trên cùng một đặc trưng, vì các mẫu khác nhau. Sự đa dạng giữa các bộ học yếu này do một *bộ tạo khác biệt* (inducer) gây ra. Một inducer ổn định tạo ra các bộ học yếu giống nhau, cho dự báo giống nhau và khó cải thiện hiệu suất mô hình. Vì vậy nên dùng inducer tương đối không ổn định, gây ra những xáo trộn nhỏ cho tập huấn luyện. Các bộ học yếu cũng đồng nhất, tức là cùng một loại mô hình, chẳng hạn đều là cây. Còn có một lợi ích khác là có thể chạy song song, vì các bộ phân loại được huấn luyện trên các mẫu độc lập. Cuối cùng, kết quả đầu ra là trung bình với hồi quy, hoặc dự báo đa số với phân loại. Việc bỏ phiếu đa số cho phân loại được gọi là bỏ phiếu cứng (hard voting) hay bỏ phiếu đa số (majority voting).

### **1.1. Bagging và Boosting**

Ta đã nhắc đến boosting ở Module 3 và sẽ đi sâu hơn ở Bài 3 và 4 của module này. Ở mức tổng quát, nên dừng lại để xem sự khác nhau giữa hai phương pháp. Hình 1 tóm tắt phương pháp của Bagging và Boosting. Bagging là quy trình song song, trong đó mỗi bộ học cơ sở không phụ thuộc bộ khác, và tập dữ liệu của mỗi bộ cũng được lấy mẫu độc lập. Boosting thì thực hiện tuần tự, mỗi bộ học cơ sở được xây dựng dựa trên bộ học trước. Với boosting, tập huấn luyện, hay các mẫu đưa vào bộ học cơ sở, được tạo từ dự báo của bộ học cơ sở trước.

**Hình 1: Bagging và Boosting**

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_4/bagging_boosting.png)

###### Biểu tượng bộ học cơ sở được thiết kế bằng tài nguyên từ Flaticon.com

Nhược điểm của bộ học kết hợp là mô hình khó diễn giải. Không rõ luồng quyết định qua các đặc trưng như ở bộ phân loại cây quyết định, vì quyết định cuối cùng đến từ nhiều bộ phân loại. Có những phương pháp như **SHAP** và **LIME** đã khắc phục vấn đề này và bỏ yếu tố hộp đen của các mô hình kết hợp; tuy nhiên các phương pháp chính xác như SHAP tốn nhiều tính toán và có thể mất nhiều thời gian, nhất là khi số quan sát cần diễn giải tăng lên. Ở phần tiếp theo, ta xem một phương pháp học tập kết hợp khác là *stacking*.

## **2. Stacking**

Hình 2: Các tập dữ liệu để xây dựng mô hình

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_4/TrainTestValid.png)

Để hiểu stacking, trước hết ta cần hiểu tập kiểm định và tập xây dựng mô hình. Hình 2 cho thấy toàn bộ dữ liệu thường được chia thành tập kiểm tra và tập huấn luyện, lần lượt ký hiệu A và B. Việc này thường làm trong Python bằng hàm `train_test_split` của `sklearn`. Ta giả sử biến mục tiêu nhị phân có nhóm 0 và 1. Để có bối cảnh, giả sử toàn bộ dữ liệu có 10.000 quan sát. Ta chia dữ liệu theo tỷ lệ 75/25 giữa huấn luyện và kiểm tra, nên A và B lần lượt có 2.500 và 7.500 quan sát. Tập huấn luyện B lại được chia tiếp theo tỷ lệ 20/80 thành tập kiểm định (C) và tập xây dựng (D), nên C có 1.500 quan sát và D có 6.000 quan sát. Tập D dùng để huấn luyện mô hình và hiệu suất mô hình được đánh giá trên C. Lưu ý tập D cũng thường được gọi là tập huấn luyện vì ta huấn luyện các bộ phân loại trên đó; tuy nhiên ta dùng từ *xây dựng* (Build) để phân biệt với B. Ta có thể dùng C và D để xây dựng mô hình vì cách này cho phép tinh chỉnh mô hình và giảm khả năng quá khớp.

Với các tập dữ liệu đó, giả sử ta huấn luyện 3 bộ phân loại như Naive Bayes (Clf1), SVM (Clf2) và cây quyết định (Clf3) trên D. Sau đó ta áp dụng các bộ phân loại này lên tập kiểm định C và thu được kết quả sau:

In [2]:
df = pd.read_csv("meta_dat2train.csv", index_col=False)
df

,Clf1,Clf2,Clf3,Y
0,1,1,0,0
1,0,0,1,1
2,0,0,0,0
3,1,1,0,0
4,0,0,1,1
...,...,...,...,...
1495,0,0,1,1
1496,0,0,0,0
1497,1,1,0,0
1498,0,0,1,1


Cột cuối "Y" là giá trị nhóm thực tế. Nhìn vào tập dữ liệu trên, ta có thể dùng các cột Clf1 đến Clf3 làm đầu vào cho một mô hình khác có biến mục tiêu Y. Mô hình ta xây dựng này gọi là mô hình *Meta* hay mô hình *Cấp 1*, và cách tiếp cận mô tả ở đây gọi là *blending*. Các tập C và D có thể được xáo trộn ngẫu nhiên thành $k$ cách chia 20/80 khác nhau. Thuật ngữ mô tả điều này là *kiểm định chéo k lần* (k-fold cross-validation), thể hiện ở Hình 3. Ta có thể xây dựng mô hình Meta trên phần thứ $k$ và dùng $k-1$ phần còn lại để huấn luyện 3 bộ phân loại Clf1-3. Cách tiếp cận trên $k$ phần này gọi là *stacking* và là điểm khác chính so với blending. Các bộ phân loại Clf1-3 gọi là mô hình cơ sở, và không nhất thiết chỉ có 3 như trong ví dụ. Bạn sẽ thấy 3 họ bộ phân loại khác nhau được dùng, khác với bagging. Điều này có nghĩa các bộ học cơ sở hay bộ học riêng lẻ trong stacking có thể không đồng nhất.

**Hình 3: Kiểm định chéo k lần**

![](https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_4/K-fold_cross_validation_EN_svg.png)

###### Nguồn: Gufosowa, Wikimedia Commons, tháng 9/2019, https://commons.wikimedia.org/wiki/File:K-fold_cross_validation_EN.svg.

## **3. Kết luận**

Học tập kết hợp dựa trên nguyên lý nhiều bộ học yếu kết hợp lại cho ước lượng tốt hơn một bộ học đơn lẻ. Phương pháp này thường cải thiện bài toán đánh đổi giữa độ chệch và phương sai. Ta đã nhắc đến hai phương pháp học tập kết hợp phổ biến là Bagging và Stacking. Bagging bị giới hạn ở các bộ học đồng nhất, còn stacking thì không. Ở bài tiếp theo, ta sẽ cài đặt một ví dụ cho từng phương pháp bằng Python.

**Tài liệu tham khảo**

- Ali, Kamal M., and Michael J. Pazzani. "Error Reduction through Learning Multiple Descriptions." *Machine Learning*, vol. 24, 1996, pp. 173–202.

- Opitz, David W., and Richard Maclin. "Popular Ensemble Methods: An Empirical Study." *Journal of Artificial Intelligence Research*, vol. 11, 1999, pp. 169-172.